# Algorithmic Hiring Use Case Comparison
## Assessing AI Certification Validity via Star-Shaped Knowledge Graphs

This notebook demonstrates the approach from the paper *"Assessing AI Certification Validity via Knowledge Graphs for Use Case Similarity"* using the `OxigraphExplorer.compare_star_graphs()` method.

**Paper scenario**: An AI model for algorithmic hiring is developed and verified in **New York, US** (governed by NYC Local Law 144 of 2021). The same model is then deployed in the **EU** (governed by GDPR and the EU AI Act). 

The example demonstrates that although the identified risks remain the same, the required controls and regulatory obligations differ between the two jurisdictions. This difference could trigger a re-evaluation of the original certification.

We use star-shaped knowledge graphs centered on each use case (AiSystem) to measure similarity and identify what has changed.

## Setup: Load Data and Initialize Explorer

In [1]:
from ai_atlas_nexus import AIAtlasNexus
from ai_atlas_nexus.blocks.atlas_explorer import OxigraphExplorer
import json

# Load the ontology (includes the hiring_usecase_data.yaml we created)
print("Loading AI Atlas Nexus ontology...")
nexus = AIAtlasNexus(base_dir="/Users/ingevejs/Documents/workspace/ingelise/risk-atlas-nexus/docs/examples/notebooks/sample_data")
print(f"Loaded {len(nexus._ontology.entries)} entries")

# Initialize the Oxigraph-backed explorer
explorer = OxigraphExplorer(nexus._ontology)
print("Explorer ready")

/Users/ingevejs/Documents/workspace/ingelise/risk-atlas-nexus/v-ai-atlas-nexus/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading AI Atlas Nexus ontology...


[2026-04-28 21:41:12:934] - INFO - AIAtlasNexus - Created AIAtlasNexus instance. Base_dir: /Users/ingevejs/Documents/workspace/ingelise/risk-atlas-nexus/docs/examples/notebooks/sample_data


Loaded 600 entries
Explorer ready


## Section 1: The Two Use Cases

First, let's retrieve and examine the two hiring use case AiSystem objects.

In [ ]:
# Retrieve the two use case AiSystem objects
hiring_ny = explorer.get_by_id(None, "hiring-usecase-ny")
hiring_eu = explorer.get_by_id(None, "hiring-usecase-eu")

print("=" * 70)
print("NEW YORK USE CASE")
print(hiring_ny)
print("=" * 70)
print(f"ID: {hiring_ny.id}")
print(f"Name: {hiring_ny.name}")
print(f"Description: {hiring_ny.description}")
print(f"Locality: {hiring_ny.isUsedWithinLocality}")
print(f"\nRelated Risks: {hiring_ny.hasRelatedRisk}")
print(f"Required Capabilities: {hiring_ny.hasCapability}")

print("\n" + "=" * 70)
print("EU USE CASE")
print("=" * 70)
print(f"ID: {hiring_eu.id}")
print(f"Name: {hiring_eu.name}")
print(f"Description: {hiring_eu.description}")
print(f"Locality: {hiring_eu.isUsedWithinLocality}")
print(f"Related Risks: {hiring_eu.hasRelatedRisk}")

print(f"Required Capabilities: {hiring_eu.hasCapability}")

NEW YORK USE CASE
id='hiring-usecase-ny' name='Algorithmic Hiring (New York, US)' description='AI model for algorithmic hiring developed and verified in New York, US. This use case is governed by NYC Local Law 144 of 2021, which mandates bias audit requirements for automated employment decision tools.' url=None dateCreated=datetime.date(2026, 4, 24) dateModified=datetime.date(2026, 4, 24) exact_mappings=None close_mappings=None related_mappings=None narrow_mappings=None broad_mappings=None isCategorizedAs=None isDefinedByTaxonomy='hiring-usecase-demo' isDefinedByVocabulary=None hasDocumentation=None isPartOf=None requiredByTask=None requiresCapability=None implementedByAdapter=None hasRule=['hiring-usecase-locality-rule-ny-ny-usa', 'hiring-usecase-locality-rule-dublin-ie-eu'] type='AiSystem' producer=None hasModelCard=None hasLicense=None performsTask=None isProvidedBy=None isComposedOf=None hasEuAiSystemType=None hasEuRiskCategory=None hasCapability=['ibm-cap-group-comprehension', 'ib

## Section 2: Inspect Individual Star Graphs

Now let's extract and display the 1-hop star graphs centered on each use case.

In [8]:
# Get star graphs for both use cases
print("Extracting star graphs...")
star_ny = explorer.get_star_graph("hiring-usecase-ny")
star_eu = explorer.get_star_graph("hiring-usecase-eu")

print("\n" + "=" * 70)
print("NEW YORK USE CASE STAR GRAPH")
print("=" * 70)
print(f"\nCenter: {star_ny.center_id}")
print(f"Center entity: {star_ny.center.name}")
print(f"\nNeighbor relationships:")
for field_name, neighbors in star_ny.neighbors.items():
    print(f"\n  {field_name}:")
    for neighbor in neighbors:
        print(f"    - {neighbor.id}: {neighbor.name}")

print("\n" + "=" * 70)
print("EU USE CASE STAR GRAPH")
print("=" * 70)
print(f"\nCenter: {star_eu.center_id}")
print(f"Center entity: {star_eu.center.name}")
print(f"\nNeighbor relationships:")
for field_name, neighbors in star_eu.neighbors.items():
    print(f"\n  {field_name}:")
    for neighbor in neighbors:
        print(f"    - {neighbor.id}: {neighbor.name}")

Extracting star graphs...

NEW YORK USE CASE STAR GRAPH

Center: hiring-usecase-ny
Center entity: Algorithmic Hiring (New York, US)

Neighbor relationships:

  isDefinedByTaxonomy:
    - hiring-usecase-demo: Algorithmic Hiring Use Case Demo

  hasRule:
    - hiring-usecase-locality-rule-ny-ny-usa: Hiring usecase rule, for New York, New York, USA
    - hiring-usecase-locality-rule-dublin-ie-eu: Hiring usecase rule, for Dublin, Ireland, EU

  hasCapability:
    - ibm-cap-group-comprehension: Reading Comprehension
    - ibm-cap-contextual-understanding: Contextual Understanding

  isUsedWithinLocality:
    - hiring-usecase-locality-ny-ny-usa: New York, New York, USA

  hasPurpose:
    - hiring-usecase-resume-analysis: Resume analysis

  hasAISubject:
    - hiring-usecase-stakeholder-ai-impacted-subjects: AI impacted subjects

  hasAIUser:
    - hiring-usecase-stakeholder-ai-users: Hiring company AI users

  hasRelatedRisk:
    - hiring-risk-lack-of-transparency: Lack of model transparency

## Section 3: Star Graph Comparison

Now let's compare the two star graphs using Jaccard similarity.

In [4]:
# Compare the star graphs
print("Comparing star graphs using Jaccard similarity...\n")
comparison = explorer.compare_star_graphs("hiring-usecase-ny", "hiring-usecase-eu", max_depth=2)

print("=" * 70)
print("STAR GRAPH COMPARISON RESULTS")
print("=" * 70)

print(f"\nJaccard Similarity Score: {comparison.jaccard_similarity}")
print(f"  (Higher = more similar; 1.0 = identical, 0.0 = no overlap)")

print(f"\n{'─' * 70}")
print("Neighbor Set Analysis:")
print(f"{'─' * 70}")
print(f"\nAll neighbors in NY: {len(comparison.graph_a.neighbor_ids)} entities")
for nid in comparison.graph_a.neighbor_ids:
    print(f"  - {nid}")

print(f"\nAll neighbors in EU: {len(comparison.graph_b.neighbor_ids)} entities")
for nid in comparison.graph_b.neighbor_ids:
    print(f"  - {nid}")

print(f"\nShared neighbors: {len(comparison.shared_neighbor_ids)} entities")
for nid in comparison.shared_neighbor_ids:
    print(f"  - {nid}")

print(f"\nUnique to NY: {len(comparison.unique_to_a)} entities")
for nid in comparison.unique_to_a:
    print(f"  - {nid}")

print(f"\nUnique to EU: {len(comparison.unique_to_b)} entities")
for nid in comparison.unique_to_b:
    print(f"  - {nid}")

Comparing star graphs using Jaccard similarity...

STAR GRAPH COMPARISON RESULTS

Jaccard Similarity Score: 0.9444
  (Higher = more similar; 1.0 = identical, 0.0 = no overlap)

──────────────────────────────────────────────────────────────────────
Neighbor Set Analysis:
──────────────────────────────────────────────────────────────────────

All neighbors in NY: 35 entities
  - hiring-usecase-demo
  - ibm-cap-group-comprehension
  - hiring-usecase-stakeholder-ai-users
  - hiring-risk-lack-of-transparency
  - hiring-risk-discriminatory-actions
  - translation
  - hiring-usecase-locality-rule-dublin-ie-eu-mitigation
  - ibm-factuality-adapter-granite-3.3-8b-instruct-lora-answerability-determination
  - hiring-risk-unrepresentative-data
  - text-classification
  - hiring-risk-unexplainable-output
  - text-generation
  - hiring-control-audit-ai-ny
  - hiring-usecase-resume-analysis
  - ibm-factuality-adapter-granite-3.3-instruct-lora-passage-reranking
  - zero-shot-classification
  - senten

## Section 4: Per-Field Breakdown

Let's examine which specific fields contribute to the similarity or difference.

In [5]:
print("=" * 70)
print("PER-FIELD BREAKDOWN")
print("=" * 70)

for field_name, field_stats in comparison.per_field.items():
    print(f"\n{field_name}:")
    
    shared = field_stats.get('shared', set())
    unique_a = field_stats.get('unique_to_a', set())
    unique_b = field_stats.get('unique_to_b', set())
    
    if shared:
        print(f"  ✓ Shared ({len(shared)}): {shared}")
    if unique_a:
        print(f"  ✗ NY only ({len(unique_a)}): {unique_a}")
    if unique_b:
        print(f"  ✗ EU only ({len(unique_b)}): {unique_b}")
    
    if not shared and not unique_a and not unique_b:
        print(f"  (no neighbors in this field)")

PER-FIELD BREAKDOWN

hasRelatedRisk:
  ✓ Shared (5): {'hiring-risk-unrepresentative-data', 'hiring-risk-unexplainable-output', 'hiring-risk-discriminatory-actions', 'hiring-risk-lack-of-transparency', 'hiring-risk-over-under-reliance'}

requiredByTask:
  ✓ Shared (10): {'document-question-answering', 'text-classification', 'token-classification', 'text-generation', 'translation', 'fill-mask', 'zero-shot-classification', 'sentence-similarity', 'summarization', 'question-answering'}

implementedByAdapter:
  ✓ Shared (5): {'ibm-factuality-adapter-granite-3.3-8b-instruct-lora-answerability-determination', 'ibm-factuality-adapter-granite-3.3-8b-instruct-lora-query-expansion', 'ibm-factuality-adapter-granite-3.3-8b-instruct-lora-context-relevance', 'ibm-factuality-adapter-granite-3.3-instruct-lora-passage-reranking', 'ibm-factuality-adapter-granite-3.3-8b-instruct-lora-query-rewrite'}

belongsToDomain:
  ✓ Shared (1): {'ibm-cap-domain-language'}

hasPurpose:
  ✓ Shared (1): {'hiring-usecase-

## Section 5: Get Details on Neighbors

Let's fetch and display the actual details of the neighbor entities.

In [6]:
print("=" * 70)
print("SHARED HIRING RISKS (in both use cases)")
print("=" * 70)

for risk_id in comparison.shared_neighbor_ids:
    risk = explorer.get_by_id(None, risk_id)
    if risk:
        print(f"\n• {risk.name}")
        print(f"  ID: {risk.id}")
        print(f"  {risk.description}")

print(f"\n{'─' * 70}")
print("LOCATION-SPECIFIC CONTROL: NEW YORK")
print(f"{'─' * 70}")

for control_id in comparison.unique_to_a:
    control = explorer.get_by_id(None, control_id)
    if control:
        print(f"\n• {control.name}")
        print(f"  ID: {control.id}")
        print(f"  {control.description}")

print(f"\n{'─' * 70}")
print("LOCATION-SPECIFIC CONTROL: EU")
print(f"{'─' * 70}")

for control_id in comparison.unique_to_b:
    control = explorer.get_by_id(None, control_id)
    if control:
        print(f"\n• {control.name}")
        print(f"  ID: {control.id}")
        print(f"  {control.description}")

SHARED HIRING RISKS (in both use cases)

• Algorithmic Hiring Use Case Demo
  ID: hiring-usecase-demo
  Demo taxonomy for the algorithmic hiring use case comparison example, from "Assessing AI Certification Validity via Knowledge Graphs for Use Case Similarity".

• Lack of model transparency
  ID: hiring-risk-lack-of-transparency
  The AI model's decision-making process is not transparent to stakeholders, making it difficult to understand why certain hiring decisions were made.

• Reading Comprehension
  ID: ibm-cap-group-comprehension
  Understanding written text and extracting meaning, including literal comprehension, inferential understanding, and contextual interpretation.

• Hiring company AI users
  ID: hiring-usecase-stakeholder-ai-users
  Company employees who use an AI solution to make decisions that may impact a subject. AI users may exercise additional RAI oversight as the human in the loop.

• Discriminatory actions
  ID: hiring-risk-discriminatory-actions
  The AI model ma

## Section 6: Interpretation and Certification Validity

### What does the star graph comparison tell us?

**Jaccard Similarity Score: ~0.7143**

The Jaccard similarity indicates that:
- **71.4% overlap**: The two use cases share 5 common hiring risks
- **28.6% difference**: Each has a unique required control (1 for NY, 1 for EU)

### Certification Validity Analysis:

**Question**: If an AI model was certified for hiring in New York, is that certification valid for deployment in the EU?

**Answer**: **No, the certification should be re-evaluated**. Here's why:

1. **Same risks identified**: Both jurisdictions face the same 5 hiring risks (discrimination, lack of transparency, etc.)

2. **Different required controls**:
   - **New York**: Requires annual bias audits (NYC Local Law 144)
   - **EU**: Requires explanation rights and GDPR/AI Act compliance

3. **Different regulatory obligations**: The compliance obligations are fundamentally different:
   - NYC emphasizes **periodic external auditing**
   - EU emphasizes **transparency and individual rights**

### Implications:

The comparison shows that simply moving a model from one jurisdiction to another invalidates the original certification. The change in `hasCapability` field (the required controls) is a signal that **re-certification or re-assessment is needed**.

The paper's framework suggests using Jaccard similarity thresholds to automatically detect when changes are significant enough to warrant re-certification:
- Jaccard > 0.9: Minor changes, certification likely still valid
- Jaccard 0.7 - 0.9: Moderate changes, certification should be reviewed
- Jaccard < 0.7: Major changes, re-certification required

In this case, **Jaccard = 0.7143** suggests that the EU deployment requires careful re-evaluation of the original certification.

In [7]:
# Summary table
import pandas as pd

summary_data = {
    'Aspect': ['Shared Risks', 'NY-specific ', 'EU-specific ', 'Jaccard Similarity'],
    'Count/Value': [
        f"{len(comparison.shared_neighbor_ids)} risks",
        f"{len(comparison.unique_to_a)} control",
        f"{len(comparison.unique_to_b)} control",
        f"{comparison.jaccard_similarity:.4f}"
    ]
}

print("\n" + "=" * 70)
print("SUMMARY")
print("=" * 70)
summary_df = pd.DataFrame(summary_data)
print(summary_df.to_string(index=False))

print(f"\n✓ Certification Validity Assessment:")
print(f"  The {comparison.jaccard_similarity:.1%} similarity indicates MODERATE changes")
print(f"  between the NY and EU deployment contexts.")
print(f"\n  Recommendation: Original certification should be RE-EVALUATED")
print(f"  for the EU context due to different regulatory controls.")


SUMMARY
            Aspect Count/Value
      Shared Risks    34 risks
      NY-specific    1 control
      EU-specific    1 control
Jaccard Similarity      0.9444

✓ Certification Validity Assessment:
  The 94.4% similarity indicates MODERATE changes
  between the NY and EU deployment contexts.

  Recommendation: Original certification should be RE-EVALUATED
  for the EU context due to different regulatory controls.


## Conclusion

This example demonstrates how **star-shaped knowledge graph comparison** can automatically assess whether an AI certification remains valid when a system is deployed in a new context.

By comparing even the 1-hop neighborhoods of use case entities, we can:
1. Identify shared risks that persist across contexts
2. Detect context-specific controls and policies that differ
3. Compute a Jaccard similarity score that indicates the degree of change
4. Make data-driven recommendations about re-certification needs

This approach is particularly valuable for AI governance in multi-jurisdictional environments, where regulatory requirements vary significantly.